# Final Assignment: Analyzing Historical Stock/Revenue Data and Building a Dashboard

**Objective:** Extract historical stock prices and quarterly revenue for Tesla (TSLA) and GameStop (GME), then visualize each company's stock price and revenue.

## Exercises
1. Extract Tesla stock data using `yfinance`, reset the index, and display the first five rows.
2. Scrape Tesla quarterly revenue and display the last five rows.
3. Extract GameStop stock data using `yfinance`, reset the index, and display the first five rows.
4. Scrape GameStop quarterly revenue and display the last five rows.
5. Plot Tesla stock price and revenue using `make_graph`.
6. Plot GameStop stock price and revenue using `make_graph`.

**Important:** Run every cell from top to bottom with internet access, then save the notebook so the actual code outputs and charts are embedded. The data-source cells intentionally do not substitute fabricated demonstration data.

## Setup
Install dependencies if needed: `pip install yfinance pandas requests beautifulsoup4 plotly lxml`. The imports below are used throughout the assignment.

In [ ]:
import pandas as pd
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from io import StringIO

print("Libraries imported successfully.")

## Data extraction helper

The revenue scraper requests the company revenue page and parses its HTML table. It does not silently substitute demonstration values: if the source blocks access, the error is shown so the source can be checked.

In [ ]:
def scrape_revenue(url, company_name):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()

    # pandas.read_html parses HTML tables; BeautifulSoup is used to confirm HTML was received.
    soup = BeautifulSoup(response.text, "html.parser")
    if soup.find("html") is None:
        raise ValueError(f"No valid HTML document returned for {company_name} revenue page.")

    tables = pd.read_html(StringIO(response.text))
    revenue_table = None
    for table in tables:
        cols = [str(col).strip().lower() for col in table.columns]
        if any("revenue" in col for col in cols):
            revenue_table = table.copy()
            break

    if revenue_table is None:
        raise ValueError(f"No revenue table found on the {company_name} page.")

    revenue_table.columns = [str(col).strip() for col in revenue_table.columns]
    revenue_col = next(col for col in revenue_table.columns if "revenue" in col.lower())
    date_col = revenue_table.columns[0]
    revenue_table = revenue_table.rename(columns={date_col: "Date", revenue_col: "Revenue"})
    revenue_table = revenue_table[["Date", "Revenue"]]
    revenue_table["Revenue"] = (
        revenue_table["Revenue"].astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .str.strip()
    )
    revenue_table["Revenue"] = pd.to_numeric(revenue_table["Revenue"], errors="coerce")
    revenue_table = revenue_table.dropna(subset=["Revenue"]).reset_index(drop=True)
    return revenue_table

## Exercise 1 — Extract Tesla stock data

Retrieve Tesla history directly with `yfinance`, reset the index so `Date` becomes a column, and display the first five rows.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Exercise 2 — Extract Tesla quarterly revenue

Scrape the Tesla revenue page, then display the last five rows using `tail()`.

In [ ]:
tesla_revenue_url = "https://www.macrotrends.net/stocks/charts/TSLA/tesla/revenue"
tesla_revenue = scrape_revenue(tesla_revenue_url, "Tesla")
display(tesla_revenue.tail())

## Exercise 3 — Extract GameStop stock data

Retrieve GameStop history directly with `yfinance`, reset the index, and display the first five rows.

In [ ]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Exercise 4 — Extract GameStop quarterly revenue

Scrape the GameStop revenue page and display the last five rows.

In [ ]:
gme_revenue_url = "https://www.macrotrends.net/stocks/charts/GME/gamestop/revenue"
gme_revenue = scrape_revenue(gme_revenue_url, "GameStop")
display(gme_revenue.tail())

## Dashboard function

The `make_graph` function creates two plots: historical stock closing price and quarterly revenue. Each plot has a descriptive title and axis labels.

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    stock_frame = stock_data.copy()
    revenue_frame = revenue_data.copy()

    stock_frame["Date"] = pd.to_datetime(stock_frame["Date"])
    revenue_frame["Date"] = pd.to_datetime(revenue_frame["Date"], errors="coerce")
    revenue_frame["Revenue"] = pd.to_numeric(revenue_frame["Revenue"], errors="coerce")
    stock_frame = stock_frame.dropna(subset=["Date", "Close"])
    revenue_frame = revenue_frame.dropna(subset=["Date", "Revenue"])

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False,
        subplot_titles=(f"{stock} - Historical Share Price",
                        f"{stock} - Historical Revenue"),
        vertical_spacing=0.15
    )
    fig.add_trace(
        go.Scatter(x=stock_frame["Date"], y=stock_frame["Close"],
                   mode="lines", name="Share price"),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=revenue_frame["Date"], y=revenue_frame["Revenue"],
                   mode="lines", name="Revenue"),
        row=2, col=1
    )
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Share price (USD)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue (USD, millions as reported)", row=2, col=1)
    fig.update_layout(
        title_text=f"{stock} Stock Price and Revenue Dashboard",
        height=750, template="plotly_white",
        showlegend=True
    )
    fig.show()
    return fig

## Exercise 5 — Plot Tesla stock data and revenue

Run `make_graph` to display both Tesla charts in the notebook output.

In [ ]:
tesla_graph = make_graph(tesla_data, tesla_revenue, "Tesla")

## Exercise 6 — Plot GameStop stock data and revenue

Run `make_graph` to display both GameStop charts in the notebook output.

In [ ]:
gme_graph = make_graph(gme_data, gme_revenue, "GameStop")

## Conclusion

This assignment retrieves historical stock prices with `yfinance`, scrapes quarterly revenue from company revenue pages, and visualizes the two datasets with Plotly. Before submitting, run all cells in order and confirm that:
- `tesla_data.head()` displays five records after `reset_index()`;
- `tesla_revenue.tail()` displays the last five scraped revenue records;
- `gme_data.head()` displays five records after `reset_index()`;
- `gme_revenue.tail()` displays the last five scraped revenue records;
- both `make_graph` calls display their charts.

If a revenue page blocks requests or changes its HTML layout, inspect the response and use the course-approved source rather than replacing it with invented data.

## References
- Yahoo Finance: https://finance.yahoo.com/
- yfinance documentation: https://ranaroussi.github.io/yfinance/
- Beautiful Soup documentation: https://www.crummy.com/software/BeautifulSoup/bs4/doc/
- Plotly Python documentation: https://plotly.com/python/